In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import os

repo_path = "/content/drive/MyDrive/Fake_Review_Ring_Detection-"

anonymized_path = os.path.join(
    repo_path,
    "data",
    "processed",
    "jumia_reviews_anonymized.csv"
)

df = pd.read_csv(anonymized_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
# Recreate the working dataframe
df_work = df.copy()

# Convert review date
df_work['review_date'] = pd.to_datetime(
    df_work['review_date'],
    errors='coerce'
)

# Create review day
df_work['review_day'] = df_work['review_date'].dt.date

print("Working dataset:", df_work.shape)

In [ ]:
# Reviewer daily review count
df_work['reviewer_daily_count'] = (
    df_work.groupby(['reviewer_id', 'review_day'])['review_id']
    .transform('count')
)

# Product daily review count
df_work['product_daily_count'] = (
    df_work.groupby(['product_id', 'review_day'])['review_id']
    .transform('count')
)

# Thresholds established yesterday
reviewer_frequency_threshold = (
    df_work['reviewer_daily_count'].quantile(0.95)
)

product_activity_threshold = (
    df_work['product_daily_count'].quantile(0.95)
)

# Temporal flags
df_work['high_reviewer_frequency'] = (
    df_work['reviewer_daily_count']
    >= reviewer_frequency_threshold
)

df_work['high_product_daily_activity'] = (
    df_work['product_daily_count']
    >= product_activity_threshold
)

# Reviewer total reviews
reviewer_total_reviews = (
    df_work.groupby('reviewer_id')['review_id']
    .transform('count')
)

df_work['reviewer_daily_concentration'] = (
    df_work['reviewer_daily_count'] /
    reviewer_total_reviews
)

# Reviewer review counts
reviewer_review_counts = (
    df_work.groupby('reviewer_id')['review_id']
    .transform('count')
)

# High daily concentration
df_work['high_reviewer_concentration'] = (
    (reviewer_review_counts >= 5) &
    (df_work['reviewer_daily_concentration'] >= 0.20)
)

# Combine temporal signals
temporal_flags = [
    'high_reviewer_frequency',
    'high_product_daily_activity',
    'high_reviewer_concentration'
]

df_work['temporal_signal_count'] = (
    df_work[temporal_flags].sum(axis=1)
)

df_work['temporal_candidate_signal'] = (
    df_work['temporal_signal_count'] >= 1
)

print("Reviewer frequency threshold:",
      reviewer_frequency_threshold)

print("Product activity threshold:",
      product_activity_threshold)

print("\nTemporal signal distribution:")
print(df_work['temporal_signal_count'].value_counts().sort_index())

print("\nReviews with at least one temporal signal:",
      df_work['temporal_candidate_signal'].sum())

In [ ]:
# Product-level rating statistics
product_rating_stats = (
    df_work.groupby('product_id')['rating']
    .agg(
        product_avg_rating='mean',
        product_rating_std='std',
        product_review_count='count'
    )
    .reset_index()
)

df_work = df_work.merge(
    product_rating_stats,
    on='product_id',
    how='left'
)

# Rating deviation from the product average
df_work['rating_deviation'] = (
    df_work['rating'] -
    df_work['product_avg_rating']
).abs()

df_work['rating_deviation_std'] = (
    df_work['rating_deviation'] /
    df_work['product_rating_std'].replace(0, pd.NA)
)

# Reviewer-level rating statistics
reviewer_rating_stats = (
    df_work.groupby('reviewer_id')['rating']
    .agg(
        reviewer_avg_rating='mean',
        reviewer_review_count='count'
    )
    .reset_index()
)

df_work = df_work.merge(
    reviewer_rating_stats,
    on='reviewer_id',
    how='left'
)

# Proportion of 5-star ratings for each reviewer
reviewer_five_star = (
    df_work.groupby('reviewer_id')['rating']
    .apply(lambda x: (x == 5).mean())
    .reset_index(name='five_star_ratio')
)

df_work = df_work.merge(
    reviewer_five_star,
    on='reviewer_id',
    how='left'
)

# Consistent 5-star reviewer signal
df_work['consistent_five_star_reviewer'] = (
    (df_work['reviewer_review_count'] >= 5) &
    (df_work['five_star_ratio'] == 1.0)
)

# Repeated reviewer-product relationship
reviewer_product_counts = (
    df_work.groupby(
        ['reviewer_id', 'product_id']
    )['review_id'].transform('count')
)

df_work['repeated_reviewer_product'] = (
    reviewer_product_counts >= 2
)

# High rating deviation
df_work['high_rating_deviation'] = (
    df_work['rating_deviation_std'] >= 2
)

# Repeated high-rating reviewer-product relationship
reviewer_product_avg_rating = (
    df_work.groupby(
        ['reviewer_id', 'product_id']
    )['rating'].transform('mean')
)

df_work['repeated_high_rating_relationship'] = (
    (reviewer_product_counts >= 2) &
    (reviewer_product_avg_rating >= 4.5)
)

# Combine rating signals
rating_signal_flags = [
    'high_rating_deviation',
    'consistent_five_star_reviewer',
    'repeated_high_rating_relationship'
]

df_work['rating_signal_count'] = (
    df_work[rating_signal_flags].sum(axis=1)
)

df_work['rating_candidate_signal'] = (
    df_work['rating_signal_count'] >= 1
)

print("Rating signal distribution:")
print(
    df_work['rating_signal_count']
    .value_counts()
    .sort_index()
)

print("\nReviews with at least one rating signal:",
      df_work['rating_candidate_signal'].sum())

In [ ]:
print("High rating deviation:",
      df_work['high_rating_deviation'].sum())

print("Consistent five-star reviewer:",
      df_work['consistent_five_star_reviewer'].sum())

print("Repeated high-rating relationship:",
      df_work['repeated_high_rating_relationship'].sum())

print("\nRepeated reviewer-product:",
      df_work['repeated_reviewer_product'].sum())

print("\nRating signal combinations:")
print(
    df_work[
        [
            'high_rating_deviation',
            'consistent_five_star_reviewer',
            'repeated_high_rating_relationship'
        ]
    ]
    .value_counts()
    .sort_index()
)

In [ ]:
# Restore the rating signal definition used yesterday
df_work['repeated_high_rating_relationship'] = (
    df_work['repeated_reviewer_product']
)

# Recalculate combined rating signals
rating_signal_flags = [
    'high_rating_deviation',
    'consistent_five_star_reviewer',
    'repeated_high_rating_relationship'
]

df_work['rating_signal_count'] = (
    df_work[rating_signal_flags].sum(axis=1)
)

df_work['rating_candidate_signal'] = (
    df_work['rating_signal_count'] >= 1
)

print("Rating signal distribution:")
print(
    df_work['rating_signal_count']
    .value_counts()
    .sort_index()
)

print("\nReviews with at least one rating signal:",
      df_work['rating_candidate_signal'].sum())

In [ ]:
repeated_reviews = df_work[
    df_work['repeated_reviewer_product']
].copy()

print("Repeated reviewer-product reviews:",
      len(repeated_reviews))

print("\nRating distribution among repeated relationships:")
print(
    repeated_reviews['rating']
    .value_counts()
    .sort_index()
)

print("\nRepeated relationship reviews by rating:")
print(
    repeated_reviews.groupby('rating').size()
)

print("\nRepeated relationship + 5-star reviews:",
      (repeated_reviews['rating'] == 5).sum())

print("\nRepeated relationship + rating >= 4:",
      (repeated_reviews['rating'] >= 4).sum())

print("\nRepeated relationship + rating >= 4.5:",
      (repeated_reviews['rating'] >= 4.5).sum())

In [ ]:
# Calculate average rating for each reviewer-product relationship
reviewer_product_avg = (
    df_work.groupby(
        ['reviewer_id', 'product_id']
    )['rating']
    .mean()
    .reset_index(name='relationship_avg_rating')
)

# Calculate number of reviews in each reviewer-product relationship
reviewer_product_count = (
    df_work.groupby(
        ['reviewer_id', 'product_id']
    )['review_id']
    .count()
    .reset_index(name='relationship_review_count')
)

# Combine relationship statistics
relationship_stats = reviewer_product_count.merge(
    reviewer_product_avg,
    on=['reviewer_id', 'product_id'],
    how='left'
)

# Select repeated relationships with a perfect average rating
selected_relationships = relationship_stats[
    (relationship_stats['relationship_review_count'] >= 2) &
    (relationship_stats['relationship_avg_rating'] == 5.0)
]

# Create keys for the selected reviewer-product relationships
selected_relationship_keys = set(
    zip(
        selected_relationships['reviewer_id'],
        selected_relationships['product_id']
    )
)

# Map selected relationships back to individual reviews
df_work['repeated_high_rating_relationship'] = [
    (reviewer_id, product_id) in selected_relationship_keys
    for reviewer_id, product_id
    in zip(df_work['reviewer_id'], df_work['product_id'])
]

print("Repeated relationships (>=2 reviews):",
      len(relationship_stats[
          relationship_stats['relationship_review_count'] >= 2
      ]))

print("Selected relationships (>=2 reviews, average rating = 5):",
      len(selected_relationships))

print("Reviews mapped to selected relationships:",
      df_work['repeated_high_rating_relationship'].sum())

print("Reviewers involved:",
      selected_relationships['reviewer_id'].nunique())

print("Products involved:",
      selected_relationships['product_id'].nunique())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import os

repo_path = "/content/drive/MyDrive/Fake_Review_Ring_Detection-"

anonymized_path = os.path.join(
    repo_path,
    "data",
    "processed",
    "jumia_reviews_anonymized.csv"
)

df = pd.read_csv(anonymized_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
import json
import os

notebook_path = (
    "/content/drive/MyDrive/Fake_Review_Ring_Detection-/"
    "notebooks/01_data_collection.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

code_cells = [
    cell for cell in notebook["cells"]
    if cell["cell_type"] == "code"
]

print("Total cells:", len(notebook["cells"]))
print("Code cells:", len(code_cells))

print("\n--- Matching cells ---")

keywords = [
    "repeated_reviewer_product",
    "repeated_high_rating_relationship",
    "stage1_candidate",
    "community_edges"
]

for i, cell in enumerate(code_cells):
    source = "".join(cell["source"])

    if any(keyword in source for keyword in keywords):
        print(f"\n===== CODE CELL {i} =====")
        print(source)

In [ ]:
import os

notebook_path = (
    "/content/drive/MyDrive/Fake_Review_Ring_Detection-/"
    "notebooks/01_data_collection.ipynb"
)

print("Checking notebook...")
print("Path:", notebook_path)
print("Exists:", os.path.exists(notebook_path))

if os.path.exists(notebook_path):
    print("File size:", os.path.getsize(notebook_path), "bytes")
else:
    print("\nNotebook was not found at this path.")
    print("\nFiles in notebooks folder:")

    notebooks_folder = (
        "/content/drive/MyDrive/Fake_Review_Ring_Detection-/notebooks"
    )

    if os.path.exists(notebooks_folder):
        print(os.listdir(notebooks_folder))
    else:
        print("The notebooks folder was not found.")

In [ ]:
import os

notebooks_folder = (
    "/content/drive/MyDrive/Fake_Review_Ring_Detection-/notebooks"
)

print("Files in notebooks folder:")
for file in os.listdir(notebooks_folder):
    file_path = os.path.join(notebooks_folder, file)
    print(f"- {file} | {os.path.getsize(file_path)} bytes")

In [ ]:
import json
import os

notebook_path = (
    "/content/drive/MyDrive/Fake_Review_Ring_Detection-/"
    "notebooks/01_data_collection.ipynb"
)

notebook = {
    "cells": [],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 5
}

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=2)

print("Notebook recreated successfully.")
print("File size:", os.path.getsize(notebook_path), "bytes")

In [ ]:
import json
import os

repo_path = "/content/drive/MyDrive/Fake_Review_Ring_Detection-"
notebook_path = os.path.join(
    repo_path, "notebooks", "01_data_collection.ipynb"
)

# The code we want to save
code = '''import pandas as pd
import os

repo_path = "/content/drive/MyDrive/Fake_Review_Ring_Detection-"

anonymized_path = os.path.join(
    repo_path,
    "data",
    "processed",
    "jumia_reviews_anonymized.csv"
)

df = pd.read_csv(anonymized_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
'''

# Read current notebook
with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Add the cell
notebook["cells"].append({
    "cell_type": "code",
    "execution_count": None,
    "metadata": {},
    "outputs": [],
    "source": code.splitlines(True)
})

# Save notebook
with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=2)

print("Dataset-loading code saved to notebook.")
print("Notebook size:", os.path.getsize(notebook_path), "bytes")

In [ ]:
import pandas as pd
import os

repo_path = "/content/drive/MyDrive/Fake_Review_Ring_Detection-"

anonymized_path = os.path.join(
    repo_path,
    "data",
    "processed",
    "jumia_reviews_anonymized.csv"
)

df = pd.read_csv(anonymized_path)

df_work = df.copy()

df_work['review_date'] = pd.to_datetime(
    df_work['review_date'],
    errors='coerce'
)

df_work['review_day'] = df_work['review_date'].dt.date

print("Working dataset:", df_work.shape)
print("Reviewers:", df_work['reviewer_id'].nunique())
print("Products:", df_work['product_id'].nunique())

In [ ]:
# Count reviews made by each reviewer on each day
df_work['reviewer_daily_count'] = (
    df_work.groupby(
        ['reviewer_id', 'review_day']
    )['review_id'].transform('count')
)

# Count reviews received by each product on each day
df_work['product_daily_count'] = (
    df_work.groupby(
        ['product_id', 'review_day']
    )['review_id'].transform('count')
)

# Determine thresholds from the dataset
reviewer_frequency_threshold = (
    df_work['reviewer_daily_count'].quantile(0.95)
)

product_activity_threshold = (
    df_work['product_daily_count'].quantile(0.95)
)

# Create temporal flags
df_work['high_reviewer_frequency'] = (
    df_work['reviewer_daily_count']
    >= reviewer_frequency_threshold
)

df_work['high_product_daily_activity'] = (
    df_work['product_daily_count']
    >= product_activity_threshold
)

print("Reviewer frequency threshold:",
      reviewer_frequency_threshold)

print("Product daily activity threshold:",
      product_activity_threshold)

print("\nHigh reviewer-frequency reviews:",
      df_work['high_reviewer_frequency'].sum())

print("High product daily-activity reviews:",
      df_work['high_product_daily_activity'].sum())

In [ ]:
# Total number of reviews made by each reviewer
reviewer_total_reviews = (
    df_work.groupby('reviewer_id')['review_id'].transform('count')
)

# Proportion of a reviewer's reviews made on that particular day
df_work['reviewer_daily_concentration'] = (
    df_work['reviewer_daily_count'] /
    reviewer_total_reviews
)

# Minimum reviewer activity required for this indicator
reviewer_review_counts = reviewer_total_reviews

# Flag high daily concentration
df_work['high_reviewer_concentration'] = (
    (reviewer_review_counts >= 5) &
    (df_work['reviewer_daily_concentration'] >= 0.20)
)

# Combine the three temporal indicators
temporal_flags = [
    'high_reviewer_frequency',
    'high_product_daily_activity',
    'high_reviewer_concentration'
]

df_work['temporal_signal_count'] = (
    df_work[temporal_flags].sum(axis=1)
)

print("Reviews with high reviewer concentration:",
      df_work['high_reviewer_concentration'].sum())

print("\nTemporal signal distribution:")
print(df_work['temporal_signal_count'].value_counts().sort_index())

print("\nReviews with at least one temporal signal:",
      (df_work['temporal_signal_count'] >= 1).sum())

In [ ]:
# Calculate rating statistics for each product
product_rating_stats = (
    df_work.groupby('product_id')['rating']
    .agg(
        product_avg_rating='mean',
        product_rating_std='std',
        product_review_count='count'
    )
    .reset_index()
)

# Add product-level statistics to each review
df_work = df_work.merge(
    product_rating_stats,
    on='product_id',
    how='left'
)

# Calculate how far each review rating is from its product's average
df_work['rating_deviation'] = (
    df_work['rating'] - df_work['product_avg_rating']
).abs()

# Standardize the deviation using the product's rating variation
df_work['rating_deviation_std'] = (
    df_work['rating_deviation'] /
    df_work['product_rating_std'].replace(0, pd.NA)
)

# Flag unusually large rating deviations
df_work['high_rating_deviation'] = (
    df_work['rating_deviation_std'] >= 2
)

print("Products:", len(product_rating_stats))

print("Reviews with high rating deviation:",
      df_work['high_rating_deviation'].sum())

print("\nProduct average rating:")
print(product_rating_stats['product_avg_rating'].describe())

In [ ]:
print("Rating data type:", df_work['rating'].dtype)

print("\nRating distribution:")
print(df_work['rating'].value_counts().sort_index())

print("\nOverall average rating:",
      df_work['rating'].mean())

print("\nProduct average rating — first 10:")
print(
    product_rating_stats[
        ['product_id', 'product_avg_rating', 'product_rating_std',
         'product_review_count']
    ].head(10)
)

In [ ]:
# Total reviews made by each reviewer
reviewer_review_count = (
    df_work.groupby('reviewer_id')['review_id'].transform('count')
)

# Number of 5-star reviews made by each reviewer
reviewer_five_star_count = (
    df_work.groupby('reviewer_id')['rating']
    .transform(lambda x: (x == 5).sum())
)

# Proportion of each reviewer's reviews that are 5-star
df_work['five_star_ratio'] = (
    reviewer_five_star_count / reviewer_review_count
)

df_work['reviewer_review_count'] = reviewer_review_count

# Flag reviewers with at least 5 reviews and all of them rated 5 stars
df_work['consistent_five_star_reviewer'] = (
    (df_work['reviewer_review_count'] >= 5) &
    (df_work['five_star_ratio'] == 1.0)
)

print("Reviewers with at least 5 reviews:",
      df_work.loc[
          df_work['reviewer_review_count'] >= 5,
          'reviewer_id'
      ].nunique())

print("Reviews from consistent 5-star reviewers:",
      df_work['consistent_five_star_reviewer'].sum())

print("Number of consistent 5-star reviewers:",
      df_work.loc[
          df_work['consistent_five_star_reviewer'],
          'reviewer_id'
      ].nunique())

In [ ]:
# Calculate average rating for each reviewer-product relationship
reviewer_product_avg = (
    df_work.groupby(
        ['reviewer_id', 'product_id']
    )['rating']
    .mean()
    .reset_index(name='relationship_avg_rating')
)

# Count reviews for each reviewer-product relationship
reviewer_product_count = (
    df_work.groupby(
        ['reviewer_id', 'product_id']
    )['review_id']
    .count()
    .reset_index(name='relationship_review_count')
)

# Combine count and average rating
relationship_stats = reviewer_product_count.merge(
    reviewer_product_avg,
    on=['reviewer_id', 'product_id'],
    how='left'
)

# Select repeated relationships with an average rating of exactly 5
selected_relationships = relationship_stats[
    (relationship_stats['relationship_review_count'] >= 2) &
    (relationship_stats['relationship_avg_rating'] == 5.0)
]

# Create lookup keys
selected_relationship_keys = set(
    zip(
        selected_relationships['reviewer_id'],
        selected_relationships['product_id']
    )
)

# Map the selected relationships back to individual reviews
df_work['repeated_high_rating_relationship'] = [
    (reviewer_id, product_id) in selected_relationship_keys
    for reviewer_id, product_id
    in zip(
        df_work['reviewer_id'],
        df_work['product_id']
    )
]

print("Repeated reviewer-product relationships:",
      (relationship_stats['relationship_review_count'] >= 2).sum())

print("Selected relationships (average rating = 5):",
      len(selected_relationships))

print("Reviews associated with selected relationships:",
      df_work['repeated_high_rating_relationship'].sum())

print("Reviewers involved:",
      df_work.loc[
          df_work['repeated_high_rating_relationship'],
          'reviewer_id'
      ].nunique())

print("Products involved:",
      df_work.loc[
          df_work['repeated_high_rating_relationship'],
          'product_id'
      ].nunique())

In [ ]:
rating_signal_flags = [
    'high_rating_deviation',
    'consistent_five_star_reviewer',
    'repeated_high_rating_relationship'
]

df_work['rating_signal_count'] = (
    df_work[rating_signal_flags].sum(axis=1)
)

print("Rating signal distribution:")
print(
    df_work['rating_signal_count']
    .value_counts()
    .sort_index()
)

print("\nReviews with at least one rating signal:",
      (df_work['rating_signal_count'] >= 1).sum())

print("Reviews with at least two rating signals:",
      (df_work['rating_signal_count'] >= 2).sum())

In [ ]:
# Normalize review text for exact repetition detection
normalized_review_text = (
    df_work['review_text']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.lower()
)

df_work['normalized_review_text'] = normalized_review_text

# Count how many times each normalized text appears
review_text_frequency = normalized_review_text.value_counts()

df_work['review_text_frequency'] = (
    normalized_review_text.map(review_text_frequency)
)

# Flag repeated non-empty review text
df_work['repeated_review_text'] = (
    (df_work['review_text_frequency'] >= 2) &
    (df_work['normalized_review_text'] != '')
)

print("Reviews with repeated text:",
      df_work['repeated_review_text'].sum())

print(
    "Unique repeated text strings:",
    df_work.loc[
        df_work['repeated_review_text'],
        'normalized_review_text'
    ].nunique()
)

print("\nMost common repeated texts:")
print(
    review_text_frequency[
        review_text_frequency >= 2
    ].head(10)
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Prepare non-empty review text
df_work['text_for_similarity'] = (
    df_work['review_text']
    .fillna('')
    .astype(str)
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

similarity_mask = df_work['text_for_similarity'] != ''

similarity_text = df_work.loc[
    similarity_mask,
    'text_for_similarity'
]

# Character-level TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=2,
    max_features=20000
)

tfidf_matrix = tfidf_vectorizer.fit_transform(similarity_text)

print("Reviews used for similarity:", tfidf_matrix.shape[0])
print("TF-IDF features:", tfidf_matrix.shape[1])

In [ ]:
# Find reviews with high cosine similarity
nn = NearestNeighbors(
    n_neighbors=20,
    metric='cosine',
    n_jobs=-1
)

nn.fit(tfidf_matrix)

distances, indices = nn.kneighbors(tfidf_matrix)

# Convert cosine distance to cosine similarity
similarity_scores = 1 - distances

print("Similarity search completed.")

print("Similarity score range:")
print(
    "Minimum:", similarity_scores.min(),
    "Maximum:", similarity_scores.max()
)

# Count non-self pairs with similarity >= 0.90
high_similarity_count = 0

for row in range(len(similarity_scores)):
    for col in range(len(similarity_scores[row])):
        if indices[row][col] != row:
            if similarity_scores[row][col] >= 0.90:
                high_similarity_count += 1

print("\nHigh-similarity neighbour relationships (>= 0.90):",
      high_similarity_count)

In [ ]:
# Get the original dataframe indices used in the similarity search
similarity_indices = df_work.index[similarity_mask].to_numpy()

near_duplicate_indices = set()

for row in range(len(similarity_scores)):
    original_index = similarity_indices[row]
    original_text = df_work.loc[
        original_index, 'text_for_similarity'
    ]

    original_length = len(original_text)

    for col in range(len(similarity_scores[row])):
        neighbour_row = indices[row][col]

        # Ignore the review matching itself
        if neighbour_row == row:
            continue

        similarity = similarity_scores[row][col]

        if similarity < 0.90:
            continue

        neighbour_index = similarity_indices[neighbour_row]
        neighbour_text = df_work.loc[
            neighbour_index, 'text_for_similarity'
        ]

        # Require both reviews to have at least 20 characters
        if original_length < 20 or len(neighbour_text) < 20:
            continue

        # Exclude exact duplicates
        if original_text == neighbour_text:
            continue

        near_duplicate_indices.add(original_index)
        near_duplicate_indices.add(neighbour_index)

df_work['near_duplicate_text'] = (
    df_work.index.isin(near_duplicate_indices)
)

print("Reviews flagged as near-duplicates:",
      df_work['near_duplicate_text'].sum())

print("Reviewers involved:",
      df_work.loc[
          df_work['near_duplicate_text'],
          'reviewer_id'
      ].nunique())

print("Products involved:",
      df_work.loc[
          df_work['near_duplicate_text'],
          'product_id'
      ].nunique())

In [ ]:
near_duplicate_pairs = []

similarity_indices = df_work.index[similarity_mask].to_numpy()

for row in range(len(similarity_scores)):
    original_index = similarity_indices[row]
    original_text = df_work.loc[
        original_index, 'text_for_similarity'
    ]

    if len(original_text) < 20:
        continue

    for col in range(len(similarity_scores[row])):
        neighbour_row = indices[row][col]

        if neighbour_row == row:
            continue

        similarity = similarity_scores[row][col]

        if similarity < 0.90:
            continue

        neighbour_index = similarity_indices[neighbour_row]
        neighbour_text = df_work.loc[
            neighbour_index, 'text_for_similarity'
        ]

        if len(neighbour_text) < 20:
            continue

        if original_text == neighbour_text:
            continue

        near_duplicate_pairs.append({
            'review_1': original_index,
            'review_2': neighbour_index,
            'similarity': similarity
        })

near_duplicate_pairs_df = pd.DataFrame(near_duplicate_pairs)

print("Near-duplicate pairs:", len(near_duplicate_pairs_df))

print("\nUnique reviews involved:",
      len(set(
          near_duplicate_pairs_df['review_1']
      ).union(
          set(near_duplicate_pairs_df['review_2'])
      )))

print("\nSimilarity distribution:")
print(
    near_duplicate_pairs_df['similarity'].describe()
)

In [ ]:
import os

processed_path = os.path.join(
    repo_path,
    "data",
    "processed"
)

os.makedirs(processed_path, exist_ok=True)

text_features_path = os.path.join(
    processed_path,
    "jumia_text_screening_features.csv"
)

text_features = df_work[
    [
        'review_id',
        'reviewer_id',
        'product_id',
        'review_text',
        'normalized_review_text',
        'review_text_frequency',
        'repeated_review_text',
        'near_duplicate_text'
    ]
].copy()

text_features.to_csv(
    text_features_path,
    index=False
)

print("Text screening features saved.")
print("Path:", text_features_path)
print("Rows:", len(text_features))
print("Columns:", len(text_features.columns))
print("File size:", os.path.getsize(text_features_path), "bytes")

In [ ]:
# Count reviews for each reviewer-product relationship
reviewer_product_counts = (
    df_work.groupby(
        ['reviewer_id', 'product_id']
    )['review_id']
    .transform('count')
)

# Flag repeated reviewer-product relationships
df_work['repeated_reviewer_product'] = (
    reviewer_product_counts >= 2
)

print("Reviews with repeated reviewer-product activity:",
      df_work['repeated_reviewer_product'].sum())

print("Reviewers involved:",
      df_work.loc[
          df_work['repeated_reviewer_product'],
          'reviewer_id'
      ].nunique())

print("Products involved:",
      df_work.loc[
          df_work['repeated_reviewer_product'],
          'product_id'
      ].nunique())

In [ ]:
# Create unique reviewer-product relationships
reviewer_product_links = (
    df_work[
        ['reviewer_id', 'product_id']
    ]
    .drop_duplicates()
)

# Group reviewers by each product
product_reviewer_groups = (
    reviewer_product_links
    .groupby('product_id')['reviewer_id']
    .apply(list)
)

# Count how many products each reviewer pair has in common
community_pair_counts = {}

for product_id, reviewers in product_reviewer_groups.items():

    reviewers = sorted(set(reviewers))

    for i in range(len(reviewers)):
        for j in range(i + 1, len(reviewers)):

            pair = (reviewers[i], reviewers[j])

            community_pair_counts[pair] = (
                community_pair_counts.get(pair, 0) + 1
            )

# Convert pair counts to a dataframe
pair_df = pd.DataFrame(
    [
        (reviewer_1, reviewer_2, shared_products)
        for (reviewer_1, reviewer_2), shared_products
        in community_pair_counts.items()
    ],
    columns=[
        'reviewer_1',
        'reviewer_2',
        'shared_products'
    ]
)

print("Reviewer-pair edges:", len(pair_df))

print("\nShared products per reviewer pair:")
print(pair_df['shared_products'].describe())

print("\nPairs sharing 2+ products:",
      (pair_df['shared_products'] >= 2).sum())

print("Pairs sharing 3+ products:",
      (pair_df['shared_products'] >= 3).sum())

print("Pairs sharing 5+ products:",
      (pair_df['shared_products'] >= 5).sum())

In [ ]:
from collections import defaultdict

# Keep only reviewer pairs sharing at least 5 products
strong_pairs = pair_df[
    pair_df['shared_products'] >= 5
].copy()

strong_pair_set = set(
    zip(
        strong_pairs['reviewer_1'],
        strong_pairs['reviewer_2']
    )
)

print("Strong reviewer pairs (5+ shared products):",
      len(strong_pair_set))

# Build reviewer -> product -> review dates
reviewer_product_dates = (
    df_work[
        ['reviewer_id', 'product_id', 'review_date']
    ]
    .dropna(subset=['review_date'])
    .groupby(
        ['reviewer_id', 'product_id']
    )['review_date']
    .apply(list)
    .to_dict()
)

# Count close same-product activities for each strong reviewer pair
close_counts = defaultdict(int)

for reviewer_a, reviewer_b in strong_pair_set:

    common_products = set()

    for reviewer_id in [reviewer_a]:
        products_a = {
            product_id
            for (rid, product_id)
            in reviewer_product_dates.keys()
            if rid == reviewer_id
        }

    products_a = {
        product_id
        for (rid, product_id)
        in reviewer_product_dates.keys()
        if rid == reviewer_a
    }

    products_b = {
        product_id
        for (rid, product_id)
        in reviewer_product_dates.keys()
        if rid == reviewer_b
    }

    common_products = products_a.intersection(products_b)

    for product_id in common_products:

        dates_a = reviewer_product_dates[
            (reviewer_a, product_id)
        ]

        dates_b = reviewer_product_dates[
            (reviewer_b, product_id)
        ]

        for date_a in dates_a:
            for date_b in dates_b:

                difference = abs(
                    (date_a - date_b).days
                )

                if difference <= 7:
                    close_counts[
                        (reviewer_a, reviewer_b)
                    ] += 1

                    break

            else:
                continue

            break

# Convert to Series
close_counts = pd.Series(
    close_counts,
    name='close_activity_count'
)

print("\nStrong pairs with close activity:",
      len(close_counts))

print("\nClose activity distribution:")
print(close_counts.describe())

print("\nPairs with 2+ close activities:",
      (close_counts >= 2).sum())

print("Pairs with 3+ close activities:",
      (close_counts >= 3).sum())

In [ ]:
from collections import defaultdict

# Count actual same-product review activities within 7 days
close_counts = defaultdict(int)

for (reviewer_a, reviewer_b) in strong_pair_set:

    # Products reviewed by both reviewers
    products_a = {
        product_id
        for (rid, product_id)
        in reviewer_product_dates.keys()
        if rid == reviewer_a
    }

    products_b = {
        product_id
        for (rid, product_id)
        in reviewer_product_dates.keys()
        if rid == reviewer_b
    }

    common_products = products_a.intersection(products_b)

    for product_id in common_products:

        dates_a = reviewer_product_dates[
            (reviewer_a, product_id)
        ]

        dates_b = reviewer_product_dates[
            (reviewer_b, product_id)
        ]

        # Count each close review-activity combination
        for date_a in dates_a:
            for date_b in dates_b:

                difference = abs(
                    (date_a - date_b).days
                )

                if difference <= 7:
                    close_counts[
                        (reviewer_a, reviewer_b)
                    ] += 1

# Convert to Series
close_counts = pd.Series(
    close_counts,
    name='close_activity_count'
)

print("Strong pairs with close activity:",
      len(close_counts))

print("\nClose activity distribution:")
print(close_counts.describe())

print("\nPairs with 2+ close activities:",
      (close_counts >= 2).sum())

print("Pairs with 3+ close activities:",
      (close_counts >= 3).sum())

In [ ]:
# Select strong reviewer pairs with at least 2 close activities
strong_close_pairs = set(
    close_counts[
        close_counts >= 2
    ].index
)

# Collect reviewers involved in these strong pairs
network_candidate_reviewers = set()

for reviewer_a, reviewer_b in strong_close_pairs:
    network_candidate_reviewers.add(reviewer_a)
    network_candidate_reviewers.add(reviewer_b)

# Map the reviewer-level signal back to individual reviews
df_work['reviewer_network_signal'] = (
    df_work['reviewer_id'].isin(
        network_candidate_reviewers
    )
)

print("Strong reviewer pairs:",
      len(strong_close_pairs))

print("Network candidate reviewers:",
      len(network_candidate_reviewers))

print("Reviews with reviewer-network signal:",
      df_work['reviewer_network_signal'].sum())

In [ ]:
# Combine the four Stage 1 signal categories

df_work['temporal_candidate_signal'] = (
    df_work['temporal_signal_count'] >= 1
)

df_work['rating_candidate_signal'] = (
    df_work['rating_signal_count'] >= 1
)

df_work['text_candidate_signal'] = (
    df_work['repeated_review_text'] |
    df_work['near_duplicate_text']
)

df_work['network_candidate_signal'] = (
    df_work['reviewer_network_signal']
)

candidate_signal_columns = [
    'temporal_candidate_signal',
    'rating_candidate_signal',
    'text_candidate_signal',
    'network_candidate_signal'
]

df_work['candidate_signal_count'] = (
    df_work[candidate_signal_columns].sum(axis=1)
)

print("Candidate signal distribution:")
print(
    df_work['candidate_signal_count']
    .value_counts()
    .sort_index()
)

print("\nReviews with at least 1 signal:",
      (df_work['candidate_signal_count'] >= 1).sum())

print("Reviews with at least 2 signals:",
      (df_work['candidate_signal_count'] >= 2).sum())

print("Reviews with at least 3 signals:",
      (df_work['candidate_signal_count'] >= 3).sum())

print("Reviews with all 4 signals:",
      (df_work['candidate_signal_count'] == 4).sum())

In [ ]:
# Stage 1 candidate selection
# A review must show at least 3 independent signal categories

df_work['stage1_candidate'] = (
    df_work['candidate_signal_count'] >= 3
)

candidate_reviews = df_work[
    df_work['stage1_candidate']
].copy()

print("Stage 1 candidate reviews:",
      len(candidate_reviews))

print("\nCandidate reviewers:",
      candidate_reviews['reviewer_id'].nunique())

print("Candidate products:",
      candidate_reviews['product_id'].nunique())

print("Candidate categories:")
print(candidate_reviews['category'].value_counts())

print("\nSignal-count distribution among candidates:")
print(
    candidate_reviews['candidate_signal_count']
    .value_counts()
    .sort_index()
)

In [ ]:
# =========================================================
# STAGE 1 — SAVE AND BACK UP
# =========================================================

import os
import json
import nbformat

# ---------------------------------------------------------
# 1. Save complete Stage 1 screening features to Drive
# ---------------------------------------------------------

screening_columns = [
    'review_id',
    'reviewer_id',
    'product_id',
    'category',
    'rating',
    'review_date',
    'review_text',

    # Temporal
    'reviewer_daily_count',
    'product_daily_count',
    'reviewer_daily_concentration',
    'high_reviewer_frequency',
    'high_product_daily_activity',
    'high_reviewer_concentration',
    'temporal_signal_count',
    'temporal_candidate_signal',

    # Rating
    'product_avg_rating',
    'product_rating_std',
    'rating_deviation',
    'rating_deviation_std',
    'high_rating_deviation',
    'reviewer_review_count',
    'five_star_ratio',
    'consistent_five_star_reviewer',
    'relationship_review_count',
    'relationship_avg_rating',
    'repeated_high_rating_relationship',
    'rating_signal_count',
    'rating_candidate_signal',

    # Text
    'normalized_review_text',
    'review_text_frequency',
    'repeated_review_text',
    'near_duplicate_text',
    'text_candidate_signal',

    # Reviewer network
    'reviewer_network_signal',
    'network_candidate_signal',

    # Final Stage 1
    'candidate_signal_count',
    'stage1_candidate'
]

screening_columns = [
    col for col in screening_columns
    if col in df_work.columns
]

screening_features_path = os.path.join(
    repo_path,
    'data',
    'processed',
    'jumia_review_screening_features.csv'
)

df_work[screening_columns].to_csv(
    screening_features_path,
    index=False
)

# ---------------------------------------------------------
# 2. Save Stage 1 candidate reviews to Drive
# ---------------------------------------------------------

candidate_path = os.path.join(
    repo_path,
    'data',
    'processed',
    'jumia_stage1_candidates.csv'
)

candidate_reviews.to_csv(
    candidate_path,
    index=False
)

# ---------------------------------------------------------
# 3. Save Stage 1 configuration
# ---------------------------------------------------------

stage1_config = {
    "dataset": "Jumia review dataset",
    "total_reviews": int(len(df_work)),

    "temporal": {
        "reviewer_frequency_percentile": 0.95,
        "reviewer_frequency_threshold": float(reviewer_frequency_threshold),
        "product_activity_percentile": 0.95,
        "product_activity_threshold": float(product_activity_threshold),
        "reviewer_concentration_min_reviews": 5,
        "reviewer_concentration_threshold": 0.20,
        "temporal_signal_minimum": 1
    },

    "rating": {
        "rating_deviation_std_threshold": 2,
        "consistent_five_star_min_reviews": 5,
        "repeated_relationship_min_reviews": 2,
        "repeated_relationship_average_rating": 5.0,
        "rating_signal_minimum": 1
    },

    "text": {
        "exact_repeat_min_frequency": 2,
        "near_duplicate_cosine_threshold": 0.90,
        "near_duplicate_min_text_length": 20,
        "exact_duplicates_excluded_from_near_duplicate": True,
        "text_signal_minimum": 1
    },

    "reviewer_network": {
        "shared_products_minimum": 5,
        "close_activity_minimum": 2,
        "temporal_window_days": 7,
        "network_signal_minimum": 1
    },

    "stage1_candidate_rule": {
        "signal_categories": [
            "temporal",
            "rating",
            "textual",
            "reviewer_network"
        ],
        "minimum_signal_categories": 3,
        "candidate_reviews": int(len(candidate_reviews)),
        "candidate_reviewers": int(
            candidate_reviews['reviewer_id'].nunique()
        ),
        "candidate_products": int(
            candidate_reviews['product_id'].nunique()
        )
    }
}

config_path = os.path.join(
    repo_path,
    'data',
    'processed',
    'stage1_thresholds.json'
)

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(stage1_config, f, indent=4)

# ---------------------------------------------------------
# 4. Save a clean copy of the notebook
# ---------------------------------------------------------

notebook_path = os.path.join(
    repo_path,
    'notebooks',
    '01_data_collection.ipynb'
)

nb = nbformat.v4.new_notebook()

# Save the important executed cells from this session.
# Temporary/empty cells are ignored.
for code in In[1:]:
    if code.strip():
        nb.cells.append(
            nbformat.v4.new_code_cell(code)
        )

with open(notebook_path, 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)

# ---------------------------------------------------------
# 5. Verify files saved to Drive
# ---------------------------------------------------------

print("=" * 60)
print("STAGE 1 FILES SAVED TO GOOGLE DRIVE")
print("=" * 60)

files_to_check = [
    screening_features_path,
    candidate_path,
    config_path,
    notebook_path
]

for path in files_to_check:
    size = os.path.getsize(path)
    print(
        f"{os.path.relpath(path, repo_path)}"
        f" -> {size:,} bytes"
    )

# ---------------------------------------------------------
# 6. Push ONLY code/configuration to GitHub
# ---------------------------------------------------------

get_ipython().run_line_magic('cd', '"$repo_path"')

get_ipython().system('git add notebooks/01_data_collection.ipynb          data/processed/stage1_thresholds.json')

get_ipython().system('git status --short')

get_ipython().system('git commit -m "Complete Stage 1 review screening"')
get_ipython().system('git push origin main')

print("\n" + "=" * 60)
print("STAGE 1 BACKUP COMPLETED")
print("=" * 60)

print("\nSaved privately in Google Drive:")
print("- jumia_review_screening_features.csv")
print("- jumia_stage1_candidates.csv")

print("\nBacked up to GitHub:")
print("- 01_data_collection.ipynb")
print("- stage1_thresholds.json")